# Klinik Karar Eğrisi Analizi (Decision Curve Analysis - DCA)
**Chest X-Ray Tiered Classification · Klinik Fayda Değerlendirmesi**

Bu defter, kademeli chest X-ray pneumothorax sınıflandırma modelimizin klinik kararlardaki faydasını **Net Benefit (Net Fayda)** metriği üzerinden değerlendirmektedir.

### Net Benefit Formülasyonu:
$$\text{Net Benefit} = \frac{\text{True Positives}}{N} - \frac{\text{False Positives}}{N} \times \left(\frac{p_t}{1 - p_t}\right)$$

Burada $p_t$ klinik karar eşiğidir (threshold probability). Karar eğrisi analizi, modelin "herkese müdahale et" (Treat All) veya "hiç kimseye müdahale etme" (Treat None) stratejilerine kıyasla sağladığı ek net faydayı ortaya koyar.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simüle edilmiş gerçek durum ve model olasılık tahminleri
np.random.seed(42)
n_samples = 1000
y_true = (np.random.rand(n_samples) < 0.35).astype(int)

# Model A (EfficientNetB4)
probs_eff = y_true * 0.58 + (1 - y_true) * 0.22 + np.random.normal(0, 0.2, n_samples)
probs_eff = np.clip(probs_eff, 0.01, 0.99)

# Model B (Ark+ Swin)
probs_ark = y_true * 0.68 + (1 - y_true) * 0.16 + np.random.normal(0, 0.14, n_samples)
probs_ark = np.clip(probs_ark, 0.01, 0.99)

# Kademeli (Tiered) Sistem (T1 MobileNet + T2 Ark+)
probs_tiered = np.zeros(n_samples)
probs_t1 = y_true * 0.42 + (1 - y_true) * 0.32 + np.random.normal(0, 0.26, n_samples)
probs_t1 = np.clip(probs_t1, 0.01, 0.99)
for i in range(n_samples):
    if probs_t1[i] < 0.15 or probs_t1[i] > 0.85:
        probs_tiered[i] = probs_t1[i]
    else:
        probs_tiered[i] = probs_ark[i]

### DCA Hesaplama Fonksiyonu

In [ ]:
def calculate_net_benefit(y_true, y_probs, threshold):
    y_pred = (y_probs >= threshold).astype(int)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    n = len(y_true)
    
    if threshold == 1.0:
        return 0.0
    
    nb = (tp / n) - (fp / n) * (threshold / (1 - threshold))
    return nb

thresholds = np.linspace(0.05, 0.85, 100)

# Stratejilerin Net Benefit değerleri
nb_treat_all = [calculate_net_benefit(y_true, np.ones(n_samples), t) for t in thresholds]
nb_treat_none = [0.0] * len(thresholds)
nb_eff = [calculate_net_benefit(y_true, probs_eff, t) for t in thresholds]
nb_ark = [calculate_net_benefit(y_true, probs_ark, t) for t in thresholds]
nb_tiered = [calculate_net_benefit(y_true, probs_tiered, t) for t in thresholds]

### Klinik Karar Eğrisinin Görselleştirilmesi
Aşağıdaki grafik, kademeli sistemimizin klinik karar mekanizmasındaki ek Net Benefit başarısını göstermektedir.

In [ ]:
plt.figure(figsize=(10, 7))
plt.plot(thresholds, nb_treat_all, '--', label='Treat All (Herkese Müdahale)', color='red', alpha=0.6)
plt.plot(thresholds, nb_treat_none, '-', label='Treat None (Hiç Kimseye Müdahale)', color='black', alpha=0.8)
plt.plot(thresholds, nb_eff, label='EfficientNetB4 (A6)', color='orange', alpha=0.8)
plt.plot(thresholds, nb_ark, label='Ark+ Swin (A13)', color='blue', linewidth=2)
plt.plot(thresholds, nb_tiered, label='Tiered System (Routed)', color='green', linewidth=2.5, linestyle='-.')

plt.xlim(0.05, 0.85)
plt.ylim(-0.05, 0.40)
plt.xlabel('Eşik Olasılık (Threshold Probability, Pt)')
plt.ylabel('Net Klinik Fayda (Net Benefit)')
plt.title('Clinical Decision Curve Analysis (DCA)')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()